# Assignment 4: Retrieval-Augmented Generation

*Published April 27, 2026*

---

## Pedagogical Purposes

Students will:
- Understand RAG applications in NLP
- Learn LangChain for building NLP applications
- Recognize RAG challenges and use cases

---

## Requirements

- Optional feedback submission via Canvas
- **Deadline:** May 28
- Submit Colab notebook, Github repo, or Python files
- Include a document indicating which tasks need feedback
- Programming-focused (no technical report required)
- Oral exam during course conclusion covering a subset of tasks

## Preliminaries

Install the required packages:

In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install langchain-core
!pip install langchain-text-splitters
!pip install sentence_transformers
!pip install langchain-chroma

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


---

## Part 1: The Dataset

### Task 1.1 — Download PubMedQA Dataset

Download the [PubMedQA dataset](https://pubmedqa.github.io/) based on medical research abstracts:

In [2]:
!wget https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json

--2026-06-09 16:36:45--  https://raw.githubusercontent.com/pubmedqa/pubmedqa/refs/heads/master/data/ori_pqal.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8000::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 2584787 (2,5M) [text/plain]
Saving to: ‘ori_pqal.json.5’

ori_pqal.json.5       0%[                    ]       0  --.-KB/s               

ori_pqal.json.5     100%[===================>]   2,46M  --.-KB/s    in 0,09s   

2026-06-09 16:36:46 (27,7 MB/s) - ‘ori_pqal.json.5’ saved [2584787/2584787]



### Collect Two Datasets

Process the downloaded file into questions and documents:

In [3]:
import pandas as pd

tmp_data = pd.read_json("ori_pqal.json").T
# some labels have been defined as "maybe", only keep the yes/no answers
tmp_data = tmp_data[tmp_data.final_decision.isin(["yes", "no"])]

documents = pd.DataFrame({
    "abstract": tmp_data.apply(lambda row: (" ").join(row.CONTEXTS + [row.LONG_ANSWER]), axis=1),
    "year": tmp_data.YEAR
})

questions = pd.DataFrame({
    "question": tmp_data.QUESTION,
    "year": tmp_data.YEAR,
    "gold_label": tmp_data.final_decision,
    "gold_context": tmp_data.LONG_ANSWER,
    "gold_document_id": documents.index
})

**Sanity check:** Inspect a sample question and document:

In [4]:
print(questions.iloc[0].question)
print(documents.iloc[0].abstract)

Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo PCD (NPCD), cells in early stages of PCD (EPCD), and cells in late stages of PCD (L

---

## Part 2: Configure LangChain LM

### Task 2.1 — Select a Language Model

Browse HuggingFace models and load one using `HuggingFacePipeline.from_model_id`.  
Set `return_full_text=False` and invoke the model with `model.invoke(your_prompt)`.

> **Note:** Gated models require a HuggingFace account and a token with *"Read access to contents of all public gated repos."*

**Sanity check:** Verify the model returns reasonable output.

In [5]:
# Task 2.1 — Load a language model from HuggingFace
# TODO: browse HuggingFace and load a model with HuggingFacePipeline.from_model_id,
#       set return_full_text=False and test with model.invoke(your_prompt).
# Note: Gemma is a gated model — run `huggingface-cli login` first with a token
#       that has "Read access to all public gated repos".

import torch
from langchain_huggingface import HuggingFacePipeline

model = HuggingFacePipeline.from_model_id(
    model_id="google/gemma-2-2b-it",
    task="text-generation",
    model_kwargs={
        "device_map": "auto",               # spreads across available GPUs
        "torch_dtype": torch.bfloat16,      # Gemma-2 is optimised for bfloat16
    },
    pipeline_kwargs={
        "max_new_tokens": 128,
        "return_full_text": False,
        "temperature": 0.1,
        "do_sample": True,
    },
)

# Sanity check
print(model.invoke("Answer in one word: What is the capital of France?"))

/home/hoda/anaconda3/envs/tch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


2026-06-09 16:36:49.692161: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Device set to use cuda:0




**Paris** 



---

## Part 3: Set Up Document Database

### Task 3.1 — Embedding Model

Use the `HuggingFaceEmbeddings` function to load an embedding model.  
Call `embed_query` and verify the output shape is `(embedding_dim,)`.

In [6]:
# Task 3.1 — Load an embedding model and verify output shape
# TODO: use HuggingFaceEmbeddings, call embed_query, verify shape is (embedding_dim,).

import torch
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Sanity check
test_emb = embedding_model.embed_query("What is programmed cell death?")
print(f"Embedding shape: ({len(test_emb)},)")
print(f"Running on: {'GPU' if torch.cuda.is_available() else 'CPU'}")

Embedding shape: (384,)
Running on: GPU


### Task 3.2 — Chunking

Use `RecursiveCharacterTextSplitter` to chunk `documents.abstract`.  
Create LangChain `Document` objects with metadata:

In [7]:
# Task 3.2 — Initialize the text splitter with your chosen chunk size / overlap
# TODO: use RecursiveCharacterTextSplitter, choose chunk_size and chunk_overlap.
#       Reflection: larger chunks preserve more context but reduce retrieval precision;
#       overlap helps avoid cutting sentences at boundaries.

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""],
)

metadatas = [{"id": idx} for idx in documents.index]
texts = text_splitter.create_documents(texts=documents.abstract.tolist(), metadatas=metadatas)

print(f"Total chunks: {len(texts)}")
print(f"\nSample chunk:\n{texts[0].page_content[:300]}")
print(f"Metadata: {texts[0].metadata}")

Total chunks: 3667

Sample chunk:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel
Metadata: {'id': 21645374}


> **Reflection:** How do chunking design choices (chunk size, overlap, splitting strategy) affect RAG quality?

I used chunk_size=512 with overlap=64. Smaller chunks help with retrieval since each chunk covers one topic, making the top-4 results more focused. The overlap avoids cutting sentences in the middle. Larger chunks would bring in irrelevant text from the same passage and confuse the model. The hit rate was 95% on the first 20 questions so this setup worked well.

### Task 3.3 — Vector Store

Use Chroma with cosine similarity.  
Apply `Chroma.from_documents` or `vector_store.add_documents` to index the chunks.  
Test retrieval with:

In [8]:
# Task 3.3 — Build the Chroma vector store
# TODO: use Chroma.from_documents with cosine similarity, index the chunks.

from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Vector store contains {vector_store._collection.count()} chunks")

Vector store contains 3667 chunks


In [9]:
# Sanity check — test similarity search
results = vector_store.similarity_search_with_score(
    "What is programmed cell death?", k=3
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.314885] Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature [{'id': 21645374}]
* [SIM=0.610320] . On the contrary, expression of the stem cell growth factor (c-kit ligand) was detected in all three uveal melanoma cell lines, suggesting the presence of autocrine (paracrine) stimulation pathways. Treatment of uveal melanoma cell lines with STI571, which blocks c-kit autophosphorylation, resulted in cell death. The IC(50) of the inhibitory effects on c-kit phosphorylation and cell proliferation was of equal size and less than 2.5 microM [{'id': 15223779}]
* [SIM=0.645404] . Separate portions of LN were snap-frozen and examin

---

## Part 4: Implementing the System

### Task 4.1 — Full RAG Pipeline

Choose **Option A** or **Option B** (not both).

---

### Option A: Agent-Based RAG

Create custom middleware that retrieves documents before each model call:

In [10]:
# Option A (skipped — using Option B below)
# The skeleton below is kept for reference only; it is not executed.

# from typing import Any
# from langchain_core.documents import Document
# from langchain.agents.middleware import AgentMiddleware, AgentState
#
# class State(AgentState):
#     context: list[Document]
#
# class RetrieveDocumentsMiddleware(AgentMiddleware[State]):
#     state_schema = State
#
#     def __init__(self, vector_store):
#         self.vector_store = vector_store
#
#     def before_model(self, state: AgentState) -> dict[str, Any] | None:
#         last_message = state["messages"][-1]
#         retrieved_docs = self.vector_store.similarity_search(last_message.text)
#         docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
#         augmented_message_content = (
#             f"Context:\n{docs_content}\n\nQuestion: {last_message.content}\n\nAnswer yes or no:"
#         )
#         return {
#             "messages": [last_message.model_copy(update={"content": augmented_message_content})],
#             "context": retrieved_docs,
#         }

print("Option A skipped — see Option B implementation below.")

Option A skipped — see Option B implementation below.


> **Hint:** Craft a prompt that instructs the model to answer yes/no classification questions.

Create an agent with `create_agent` using the middleware, then stream results:

In [11]:
# Option A test cell (skipped — using Option B)
# agent is not defined; see the Option B chain below.
print("Option A skipped — using Option B (LCEL chain).")

Option A skipped — using Option B (LCEL chain).


---

### Option B: Chain-Based RAG (LCEL)

Define a retriever from the vector store and build a chain using LangChain Expression Language:

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Task 4.1 Option B — Build the RAG chain
# TODO: define retriever, prompt template, chain, and combine with RunnableParallel.

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    "You are a medical research assistant. "
    "Use the PubMed abstracts below to answer the yes/no question.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer with only 'yes' or 'no' based on the evidence provided:"
)

# Chain: format context + question → prompt → model → string
chain = (
    {"context": lambda x: format_docs(x["context"]), "question": lambda x: x["question"]}
    | prompt
    | model
    | StrOutputParser()
)

# Run retrieval and passthrough in parallel, then assign the answer
setup = RunnableParallel(
    context=lambda x: retriever.invoke(x["question"]),
    question=lambda x: x["question"],
)
rag_chain = setup.assign(answer=chain)

**Sanity check:** Test with a sample question and verify document retrieval and answer quality:

In [13]:
# Sanity check — run the chain on a sample question
sample_question = questions.iloc[0].question
print(f"Question: {sample_question}")
print(f"Gold label: {questions.iloc[0].gold_label}\n")

answer = rag_chain.invoke({"question": sample_question})
print("Model answer:", answer["answer"])
print("\nRetrieved context (first chunk):")
print(answer["context"][0].page_content[:400])

Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Gold label: yes



Model answer: 

Yes/No 


Retrieved context (first chunk):
. Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar strands to form a ring structure surrounding the nucleus during developmental PCD. Also, for the firs


---

## Part 5: Evaluate RAG

### Task 5.1 — High-Level Evaluation

Evaluate your system on `questions.question` and `questions.gold_label` using F1 and/or accuracy metrics.  
Handle invalid/unparseable model answers separately.  
Compare RAG performance against a baseline LM **without** context.

In [14]:
# Task 5.1 — Evaluate RAG vs. baseline LM
# TODO: run on questions/gold_labels, extract yes/no, compute F1/accuracy,
#       compare RAG against a baseline LM without context.

import re
from sklearn.metrics import f1_score, accuracy_score

def extract_yes_no(text):
    """Return 'yes', 'no', or None if the answer is not parseable."""
    text = text.lower().strip()
    if text.startswith("yes"):
        return "yes"
    if text.startswith("no"):
        return "no"
    m = re.search(r'\b(yes|no)\b', text)
    return m.group(1) if m else None


# Baseline: same LM without retrieved context
baseline_prompt = ChatPromptTemplate.from_template(
    "You are a medical research assistant. "
    "Answer the following question with only 'yes' or 'no':\n\n"
    "Question: {question}\n\nAnswer:"
)
baseline_chain = baseline_prompt | model | StrOutputParser()

# Evaluate on first 100 questions (adjust N for full eval)
N = 100
eval_qs = questions.head(N)

rag_preds, rag_gold = [], []
base_preds, base_gold = [], []

for _, row in eval_qs.iterrows():
    q, label = row["question"], row["gold_label"]

    rag_raw  = rag_chain.invoke({"question": q})["answer"]
    base_raw = baseline_chain.invoke({"question": q})

    rag_ans  = extract_yes_no(rag_raw)
    base_ans = extract_yes_no(base_raw)

    if rag_ans  is not None:
        rag_preds.append(rag_ans);  rag_gold.append(label)
    if base_ans is not None:
        base_preds.append(base_ans); base_gold.append(label)

def report(name, preds, gold):
    if not preds:
        print(f"\n{name}: no parseable predictions")
        return
    acc = accuracy_score(gold, preds)
    f1  = f1_score(gold, preds, pos_label="yes", average="binary", zero_division=0)
    print(f"\n{name}  (n={len(preds)}, invalid={N - len(preds)})")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  F1 (yes) : {f1:.3f}")
    print(f"  Dist     : yes={preds.count('yes')}, no={preds.count('no')}")

report("RAG model",            rag_preds,  rag_gold)
report("Baseline (no context)", base_preds, base_gold)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



RAG model  (n=96, invalid=4)
  Accuracy : 0.750
  F1 (yes) : 0.833
  Dist     : yes=77, no=19

Baseline (no context)  (n=100, invalid=0)
  Accuracy : 0.680
  F1 (yes) : 0.805
  Dist     : yes=94, no=6


### Task 5.2 — Detailed Inspection

Compare retrieved document IDs against `questions.gold_document_id`.  
Inspect samples and assess how well the pipeline is functioning.

In [15]:
# Task 5.2 — Compare retrieved document IDs to gold document IDs
# TODO: for each question retrieve docs, check if gold_document_id is in retrieved ids,
#       compute hit rate and inspect samples.

hits = 0
INSPECT_N = 20
inspect_qs = questions.head(INSPECT_N)

print(f"{'Question (truncated)':<55} {'Gold doc':<15} {'Retrieved IDs':<40} {'Hit'}")
print("-" * 120)

for _, row in inspect_qs.iterrows():
    docs = retriever.invoke(row["question"])
    retrieved_ids = [doc.metadata["id"] for doc in docs]
    gold_id = row["gold_document_id"]
    hit = gold_id in retrieved_ids
    hits += int(hit)

    print(f"{row['question'][:53]:<55} {str(gold_id):<15} {str(retrieved_ids[:3]):<40} {'✓' if hit else '✗'}")

print(f"\nRetrieval hit rate (top-4): {hits}/{INSPECT_N} = {hits/INSPECT_N:.0%}")
print("\nInterpretation: a hit means the gold document was among the top-4 retrieved chunks.")
print("Low hit rate suggests the embedding model or chunking strategy needs improvement.")

Question (truncated)                                    Gold doc        Retrieved IDs                            Hit
------------------------------------------------------------------------------------------------------------------------
Do mitochondria play a role in remodelling lace plant   21645374        [21645374, 21645374, 21645374]           ✓
Landolt C and snellen e acuity: differences in strabi   16418930        [16418930, 16418930, 16418930]           ✓
Syncope during bathing in infants, a pediatric form o   9488747         [9488747, 9488747, 9488747]              ✓
Are the long-term results of the transanal pull-throu   17208539        [17208539, 17208539, 21481154]           ✓
Can tailored interventions increase mammography use a   10808977        [10808977, 10808977, 10808977]           ✓
Double balloon enteroscopy: is it efficacious and saf   23831910        [25251991, 17593459, 10922093]           ✗
Is adjustment for reporting heterogeneity necessary i   26852225        